# **Tập dữ liệu**

## Load Data

In [ ]:
from datasets import load_dataset

ds = load_dataset(
    "imagefolder",
    data_dir="../data/raw/intel_images",
)

In [ ]:
print(ds)

## Chuẩn bị dữ liệu

Mặc dù tập dữ liệu đã được chia sẵn thành train và test set, nhưng do yêu cầu của bài tập tập trung vào bước tiền xử lý và số lượng ảnh trong dataset khá lớn khiến thời gian tính toán bị kéo dài. Vì vậy, trong phạm vi bài tập này, nhóm chỉ chọn ngẫu nhiên 5.000 ảnh từ tập train để tiến hành xử lý và đánh giá.

In [ ]:
# Define constants
N = 5000
H = 150
W = 150
SEED = 42

In [ ]:
dataset = ds["train"].shuffle(seed=SEED).select(range(N))
len(dataset)

In [ ]:
dataset[0]

In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image

image_list = []
label_list = []
original_size = (150, 150)

for item in tqdm(dataset):
    pil_img = item["image"].convert("RGB")
    img_resized_pil = pil_img.resize(original_size, Image.LANCZOS)
    img_resized_np = np.array(img_resized_pil)

    image_list.append(img_resized_np)
    label_list.append(item["label"])

In [ ]:
images = np.array(image_list)
labels = np.array(label_list)

In [ ]:
print(f"\nShape of images: {images.shape}")  # (5000, 150, 150, 3) ~ (N, H, W, 3)
print(f"Shape of label: {labels.shape}")  # (5000,)

In [ ]:
print(label_list)

In [ ]:
class_names = dataset.features["label"].names
print(f"Classes: {class_names}")
print(f"Number of classes: {len(class_names)}")
unique_labels = np.sort(pd.Series(label_list).unique())
print(f"Labels: {unique_labels}")

---
# **Các kỹ thuật tiền xử lý và phân tích tác động**

In [ ]:
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score, StratifiedShuffleSplit
from sklearn.manifold import TSNE
from sklearn.preprocessing import LabelEncoder

from scipy.stats import ks_2samp, f_oneway
from skimage.metrics import (
    structural_similarity as ssim,
    peak_signal_noise_ratio as psnr,
)
from skimage.color import rgb2gray, rgb2hsv, rgb2lab
from skimage.filters import sobel, prewitt, sobel_h, sobel_v, prewitt_h, prewitt_v
from skimage.feature import canny
from skimage.transform import resize as sk_resize
import cv2

from matplotlib import pyplot as plt
import gc
import albumentations

In [ ]:
NUM_CLASSES = len(class_names)

In [ ]:
def stratified_sample_idx(y, n, seed=SEED):
    """Trả về CHỈ SỐ của n mẫu stratified — không copy data."""
    n = min(n, len(y))
    sss = StratifiedShuffleSplit(n_splits=1, test_size=n / len(y), random_state=seed)
    _, idx = next(sss.split(np.zeros(len(y)), y))
    return idx


def run_ablation(
    X_flat_sampled,
    y_sampled,
    label: str,
    n_pca: int = 50,
    cv: int = 5,
    k_knn: int = 5,
    C_lr: float = 1.0,
    verbose: bool = True,
) -> dict:
    X_s = X_flat_sampled.astype(np.float64)
    y_s = y_sampled

    actual_pca = 0
    if n_pca > 0:
        actual_pca = min(n_pca, X_s.shape[1], X_s.shape[0] - 1)
        pca = PCA(n_components=actual_pca, random_state=SEED)
        X_s = pca.fit_transform(X_s)  # replace, không cộng dồn
        del pca

    lr = LogisticRegression(
        C=C_lr,
        max_iter=1000,
        solver="lbfgs",
        random_state=SEED,
        n_jobs=-1,
    )
    knn = KNeighborsClassifier(n_neighbors=k_knn, n_jobs=-1)

    lr_scores = cross_val_score(lr, X_s, y_s, cv=cv, scoring="accuracy", n_jobs=-1)
    knn_scores = cross_val_score(knn, X_s, y_s, cv=cv, scoring="accuracy", n_jobs=-1)

    del X_s, lr, knn  # giải phóng ngay

    result = dict(
        label=label,
        n_sample=int(len(y_s)),
        n_pca_components=int(actual_pca),
        lr_acc=float(lr_scores.mean()),
        lr_std=float(lr_scores.std()),
        lr_scores=lr_scores,
        knn_acc=float(knn_scores.mean()),
        knn_std=float(knn_scores.std()),
        knn_scores=knn_scores,
    )

    if verbose:
        pca_tag = f"PCA-{actual_pca}" if n_pca > 0 else "no PCA"
        print(f"  [{label:<30s}]  n={len(y_s):4d}  {pca_tag}")
        print(
            f'    LogReg : {result["lr_acc"]:.4f} ± {result["lr_std"]:.4f}'
            f"  | folds: {np.round(lr_scores, 4)}"
        )
        print(
            f'    k-NN   : {result["knn_acc"]:.4f} ± {result["knn_std"]:.4f}'
            f"  | folds: {np.round(knn_scores, 4)}"
        )

    return result

In [ ]:
ABLATION_REGISTRY: list[dict] = []


def register(result: dict):
    """Lưu kết quả vào registry dict ABLATION_REGISTRY"""
    ABLATION_REGISTRY.append(result)
    return result


def ablation_summary_table(registry=None) -> pd.DataFrame:
    reg = registry or ABLATION_REGISTRY
    rows = []
    for r in reg:
        rows.append(
            {
                "Config": r["label"],
                "N sample": r["n_sample"],
                "PCA k": r["n_pca_components"],
                "LogReg acc": f"{r['lr_acc']:.4f}",
                "LogReg ±": f"{r['lr_std']:.4f}",
                "k-NN acc": f"{r['knn_acc']:.4f}",
                "k-NN ±": f"{r['knn_std']:.4f}",
            }
        )
    df = pd.DataFrame(rows)
    return df

## Ablation pipeline Arguments

In [ ]:
N_SAMPLE = 1000
N_PCA = 50
CV = 5
K_KNN = 3
C_LOGREG = 2.0

SAMPLED_INDICES = stratified_sample_idx(labels, n=N_SAMPLE, seed=SEED)
SAMPLED_LABELS = labels[SAMPLED_INDICES]

print(f"Sample: {len(SAMPLED_INDICES)} images")
classes, counts = np.unique(SAMPLED_LABELS, return_counts=True)
counts_dict = dict(zip(classes.tolist(), counts.tolist()))

print(f"class counts: {counts_dict}")

## Chạy Baseline trên Ảnh Gốc

Trước khi áp dụng bất kỳ kỹ thuật tiền xử lý nào, chúng ta cần có điểm baseline trên ảnh gốc (150×150) để so sánh.

In [ ]:
# Baseline (150×150 raw)
X_base = images[SAMPLED_INDICES].reshape(N_SAMPLE, -1).astype(np.float64)
baseline_res = run_ablation(
    X_base,
    SAMPLED_LABELS,
    label="Baseline (150x150 raw)",
    n_pca=N_PCA,
    cv=CV,
    k_knn=K_KNN,
    C_lr=C_LOGREG,
)
del X_base
register(baseline_res)

---
## (a) Thay Đổi Kích Thước Và Chất Lượng Ảnh

Thay đổi kích thước và chất lượng ảnh: Resize về ít nhất 3 kích thước khác nhau
(ví dụ: 32 × 32, 64 × 64, 128 × 128). Với mỗi kích thước, tính chỉ số SSIM (Structural Similarity Index) và PSNR so với ảnh gốc để định lượng mức độ mất mát thông tin. Vẽ đường cong SSIM theo kích thước và biện hộ cho kích thước được chọn.

### Lý thuyết

**SSIM (Structural Similarity Index)** đo mức độ tương đồng cấu trúc giữa hai ảnh $x$ và $y$:

$$\text{SSIM}(x, y) = \frac{(2\mu_x\mu_y + c_1)(2\sigma_{xy} + c_2)}{(\mu_x^2 + \mu_y^2 + c_1)(\sigma_x^2 + \sigma_y^2 + c_2)}$$

Trong đó $\mu$, $\sigma^2$, $\sigma_{xy}$ là trung bình, phương sai, và hiệp phương sai cục bộ; $c_1, c_2$ là hằng số ổn định. SSIM $\in [-1, 1]$, bằng 1 khi hai ảnh giống hệt nhau.


**PSNR (Peak Signal-to-Noise Ratio)**:

$$\text{PSNR} = 10 \cdot \log_{10}\!\left(\frac{\text{MAX}^2}{\text{MSE}}\right) \quad [\text{dB}]$$

PSNR càng cao thì mất mát thông tin càng ít. Thông thường PSNR > 30 dB được coi là chấp nhận được.

In [ ]:
TARGET_SIZES = [32, 64, 128]
resize_metrics = {}
resized_datasets = {}


def resize_dataset(imgs, size):
    """Resize mảng (N,H,W,3) -> (N,size,size,3), giá trị trong [0,1] float32."""
    out = np.zeros((len(imgs), size, size, 3), dtype=np.float32)
    for i, img in enumerate(imgs):
        out[i] = sk_resize(img, (size, size), anti_aliasing=True)
    return out

In [ ]:
for sz in TARGET_SIZES:
    resized_images = resize_dataset(images, sz)
    resized_datasets[sz] = resized_images
    restored_images = resize_dataset(
        resized_images, H
    )  # Upscale về size ban đầu để kiểm tra độ mất mát thông tin

    ssim_vals, psnr_vals = [], []

    for resz, rest, orig in zip(resized_images, restored_images, images):
        s = ssim(orig, rest, channel_axis=-1, data_range=1.0)
        p = psnr(orig, rest, data_range=1.0)
        ssim_vals.append(s)
        psnr_vals.append(p)

    resize_metrics[sz] = {
        "ssim_mean": np.mean(ssim_vals),
        "ssim_std": np.std(ssim_vals),
        "psnr_mean": np.mean(psnr_vals),
        "psnr_std": np.std(psnr_vals),
    }

    fig, ax = plt.subplots(1, 3, figsize=(24, 6))

    random_index = np.random.choice(len(images))
    ax[0].imshow(resized_images[random_index])
    ax[0].set_title(f"Ảnh resized {sz}×{sz}")

    ax[1].imshow(restored_images[random_index])
    ax[1].set_title(f"Ảnh restored về size gốc {H}×{W}")

    ax[2].imshow(images[random_index])
    ax[2].set_title(f"Ảnh gốc {H}×{W}")

    fig.suptitle(
        f"{sz}×{sz} | SSIM={np.mean(ssim_vals):.4f}±{np.std(ssim_vals):.4f} | PSNR={np.mean(psnr_vals):.2f}±{np.std(psnr_vals):.2f} dB"
    )
    plt.show()

In [ ]:
print("Ablation study theo kich thuoc anh:")
for sz in TARGET_SIZES:
    # Extract + flatten only the N_SAMPLE rows we need
    X_sz = (
        resized_datasets[sz][SAMPLED_INDICES].reshape(N_SAMPLE, -1).astype(np.float64)
    )

    res = run_ablation(
        X_sz,
        SAMPLED_LABELS,
        label=f"Resize {sz}x{sz}",
        n_pca=N_PCA,
        cv=CV,
        k_knn=K_KNN,
        C_lr=C_LOGREG,
    )

    del X_sz
    resize_metrics[sz].update(res)
    register(res)

In [ ]:
# SSIM curve, Accuracy vs. Size, Visual quality strip
szs = TARGET_SIZES
ssim_means = [resize_metrics[s]["ssim_mean"] for s in szs]
ssim_stds = [resize_metrics[s]["ssim_std"] for s in szs]
lr_accs = [resize_metrics[s]["lr_acc"] for s in szs]
knn_accs = [resize_metrics[s]["knn_acc"] for s in szs]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(
    "(a) Resize Analysis - SSIM, LogReg & k-NN Accuracy", fontsize=13, fontweight="bold"
)
kw = dict(marker="o", linewidth=2, capsize=5)

# 1. SSIM vs Size
axes[0].errorbar(szs, ssim_means, yerr=ssim_stds, color="steelblue", **kw)
axes[0].set(title="SSIM vs. Size", xlabel="Size (px)", ylabel="SSIM")
axes[0].set_xticks(szs)

# 2. Classifier Accuracy vs Size
axes[1].plot(szs, lr_accs, marker="^", linewidth=2, color="seagreen", label="LogReg")
axes[1].plot(szs, knn_accs, marker="D", linewidth=2, color="crimson", label="k-NN")
axes[1].set(
    title="Classifier Accuracy vs. Size", xlabel="Size (px)", ylabel="CV Accuracy"
)
axes[1].set_xticks(szs)
axes[1].legend()

# accuracy values
for s, v in zip(szs, lr_accs):
    axes[1].annotate(
        f"{v:.3f}",
        (s, v),
        xytext=(6, 4),
        textcoords="offset points",
        fontsize=8,
        color="seagreen",
    )
for s, v in zip(szs, knn_accs):
    axes[1].annotate(
        f"{v:.3f}",
        (s, v),
        xytext=(6, -12),
        textcoords="offset points",
        fontsize=8,
        color="crimson",
    )

# 3. Visual quality strip
strips = [sk_resize(images[0], (sz, sz), anti_aliasing=True) for sz in szs]
strip = np.concatenate(
    [sk_resize(s, (128, 128), anti_aliasing=True) for s in strips], axis=1
)
axes[2].imshow(np.clip(strip, 0, 1))
axes[2].axis("off")
axes[2].set_title("Visual Quality: " + " | ".join(map(str, szs)))

# Apply grid to the first two numerical plots
for ax in axes[:2]:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("a_resize_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

### Phân tích kết quả

- SSIM tăng theo kích thước ảnh: ảnh 128×128 giữ lại nhiều thông tin cấu trúc nhất so với ảnh gốc 150×150.
- PSNR cũng tăng tương ứng, xác nhận mức độ mất mát pixel giảm khi kích thước lớn hơn.
- Ảnh 64×64 đạt cân bằng tốt nhất giữa chất lượng (SSIM/PSNR) và hiệu quả phân loại

---
## (b) Chuyển Đổi Không Gian Màu

RGB, Grayscale, HSV, và LAB. Với mỗi không gian màu, tính phương sai giải thích
(explained variance) theo PCA với k = 50 thành phần. Thảo luận không gian màu nào bảo toàn thông tin tốt nhất cho bài toán phân loại.

### Lý thuyết

**RGB**: Mỗi kênh $\{R, G, B\} \in [0,1]$. Dễ xử lý nhưng các kênh tương quan cao.

**Grayscale**: $I = 0.2989R + 0.5870G + 0.1140B$ — nén 3 kênh → 1, giữ độ sáng.

**HSV** (Hue–Saturation–Value): Tách biệt màu sắc (H) khỏi độ sáng (V), gần với nhận thức của người hơn RGB.

**LAB**: Không gian đồng nhất về nhận thức (*perceptually uniform*). Kênh L (độ sáng) độc lập hoàn toàn với a, b (màu sắc). Khuyến nghị cho phân loại cảnh quan.

**Explained Variance (PCA)** với $k$ thành phần:
$$\text{EV}(k) = \sum_{i=1}^{k} \frac{\lambda_i}{\sum_j \lambda_j}$$
EV cao → không gian màu gọn hơn, ít dư thừa hơn.

In [ ]:
COLOR_SPACES = ["RGB", "Grayscale", "HSV", "LAB"]
cs_results = {}


def to_colorspace(imgs, mode):
    """Convert (N,H,W,3) float32 [0,1] → target color space, normalized to [0,1]."""
    out = []
    for img in imgs:
        if mode == "RGB":
            out.append(img)
        elif mode == "Grayscale":
            g = rgb2gray(img)
            out.append(np.stack([g, g, g], axis=-1))
        elif mode == "HSV":
            out.append(rgb2hsv(img))
        elif mode == "LAB":
            lab = rgb2lab(img)
            lab_norm = (lab + [0, 128, 128]) / [100, 255, 255]
            out.append(lab_norm.astype(np.float32))
    return np.array(out, dtype=np.float32)

In [ ]:
print("Chuyen doi khong gian mau va tinh phuong sai giai thich (EV):")
print(
    f'{"Khong gian mau":12s} | {"EV@k=50 (%)":>12s} | {"LR acc":>10s} | {"k-NN acc":>10s}'
)
print("-" * 55)
SAMPLE_IDX = SAMPLED_INDICES[0]
cs_samples = {}


def ensure_float01(img):
    """Đảm bảo ảnh là float32 [0,1] bất kể dtype đầu vào."""
    img = np.array(img)
    if img.dtype == np.uint8:
        return img.astype(np.float32) / 255.0
    elif img.max() > 1.0:
        return (img / img.max()).astype(np.float32)
    return img.astype(np.float32)


def convert_single(img, mode):
    """Convert 1 ảnh (H,W,3) → color space, trả về (H,W,3) float32 [0,1]."""
    img = ensure_float01(img)  # ← guard against uint8 / wrong range
    if mode == "RGB":
        return img.copy()
    elif mode == "Grayscale":
        g = rgb2gray(img)
        return np.stack([g, g, g], axis=-1).astype(np.float32)
    elif mode == "HSV":
        return rgb2hsv(img).astype(np.float32)
    elif mode == "LAB":
        lab = rgb2lab(img)
        return ((lab + [0, 128, 128]) / [100, 255, 255]).astype(np.float32)


def convert_rows_flat(imgs, idx, mode):
    """Convert chỉ những hàng trong idx."""
    H_, W_, C_ = imgs[0].shape
    out = np.empty((len(idx), H_ * W_ * C_), dtype=np.float64)
    for i, src_i in enumerate(idx):
        out[i] = convert_single(imgs[src_i], mode).ravel()
    return out


for cs in COLOR_SPACES:
    cs_samples[cs] = convert_single(images[SAMPLE_IDX], cs)

    X_cs_flat = convert_rows_flat(images, SAMPLED_INDICES, cs)
    pca_ev = PCA(random_state=SEED).fit(X_cs_flat)
    cum_var = np.cumsum(pca_ev.explained_variance_ratio_) * 100
    ev50 = float(cum_var[min(49, len(cum_var) - 1)])
    del pca_ev

    res = run_ablation(
        X_cs_flat,
        SAMPLED_LABELS,
        label=f"ColorSpace {cs}",
        n_pca=N_PCA,
        cv=CV,
        k_knn=K_KNN,
        C_lr=C_LOGREG,
    )
    del X_cs_flat

    cs_results[cs] = {**res, "ev50": ev50, "cum_var": cum_var}
    register(res)
    print(f'  {cs:10s} | {ev50:12.2f} | {res["lr_acc"]:10.4f} | {res["knn_acc"]:10.4f}')

# ── Visual comparison ─────────────────────────────────────────
CHANNEL_HINTS = {
    "RGB": {"cmap": None, "note": "Hiển thị trực tiếp"},
    "Grayscale": {"cmap": "gray", "note": "Kênh độ sáng (L)"},
    "HSV": {"cmap": None, "note": "H=hue  S=sat  V=val"},
    "LAB": {"cmap": None, "note": "L=sáng  a/b=màu (norm)"},
}
n_cols = len(COLOR_SPACES) + 1
fig, axes = plt.subplots(1, n_cols, figsize=(5 * n_cols, 5))
fig.suptitle(
    f"Minh họa chuyển đổi không gian màu (ảnh #{SAMPLE_IDX})",
    fontsize=13,
    fontweight="bold",
)

axes[0].imshow(ensure_float01(images[SAMPLE_IDX]))
axes[0].set_title("Original\n(RGB)", fontsize=11, fontweight="bold")
axes[0].axis("off")

for i, cs in enumerate(COLOR_SPACES):
    ax = axes[i + 1]
    img = cs_samples[cs]
    if cs == "Grayscale":
        ax.imshow(img[:, :, 0], cmap="gray", vmin=0, vmax=1)
    else:
        ax.imshow(img, vmin=0, vmax=1)
    ev = cs_results[cs]["ev50"]
    acc = cs_results[cs]["lr_acc"]
    ax.set_title(
        f'{cs}\n{CHANNEL_HINTS[cs]["note"]}\nEV@50={ev:.1f}%  LR={acc:.3f}', fontsize=9
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
colors_cs = {
    "RGB": "steelblue",
    "Grayscale": "dimgray",
    "HSV": "darkorange",
    "LAB": "mediumpurple",
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("(b) So sanh cac khong gian mau", fontsize=13, fontweight="bold")

for cs in COLOR_SPACES:
    cum = cs_results[cs]["cum_var"]
    axes[0].plot(
        range(1, min(201, len(cum) + 1)),
        cum[:200],
        label=cs,
        color=colors_cs[cs],
        linewidth=2,
    )
axes[0].axhline(90, color="red", linestyle="--", alpha=0.6, label="90%")
axes[0].axhline(95, color="green", linestyle="--", alpha=0.6, label="95%")
axes[0].set(
    title="Cumulative Explained Variance (PCA)",
    xlabel="# Components",
    ylabel="Cumulative Variance (%)",
)
axes[0].set_xlim(0, 200)
axes[0].set_ylim(0, 101)
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.3)

# EV@50 bar
x = np.arange(len(COLOR_SPACES))
w = 0.4
ev_vals = [cs_results[cs]["ev50"] for cs in COLOR_SPACES]
lr_vals = [cs_results[cs]["lr_acc"] * 100 for cs in COLOR_SPACES]
knn_vals = [cs_results[cs]["knn_acc"] * 100 for cs in COLOR_SPACES]

axes[1].bar(x, ev_vals, w, color=[colors_cs[c] for c in COLOR_SPACES], alpha=0.85)
axes[1].set_xticks(x)
axes[1].set_xticklabels(COLOR_SPACES)
axes[1].set(title="Explained Variance @ k=50", ylabel="EV (%)")
axes[1].grid(True, alpha=0.3, axis="y")

axes[2].bar(
    x - w / 2,
    lr_vals,
    w,
    label="LogReg",
    color=[colors_cs[c] for c in COLOR_SPACES],
    alpha=0.85,
)
axes[2].bar(
    x + w / 2,
    knn_vals,
    w,
    label="k-NN",
    color=[colors_cs[c] for c in COLOR_SPACES],
    alpha=0.5,
    hatch="//",
)
axes[2].set_xticks(x)
axes[2].set_xticklabels(COLOR_SPACES)
axes[2].set(title="Classifier Accuracy per Color Space", ylabel="CV Accuracy (%)")
axes[2].legend()
axes[2].grid(True, alpha=0.3, axis="y")

plt.tight_layout()
# plt.savefig('b_colorspace_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
del cs_results
del cs_samples

### Phân tích kết quả

- Ở đồ thị cumulative EV cả 4 đường cong cho 4 không gian màu đều tăng nhanh ở những thành phần đầu tiên và thoải dần sau đó. Với 200 thành phần, chưa có không gian màu nào đạt tới ngưỡng giải thích 90% phương sai (đường nét đứt màu đỏ). Điều này cho thấy tập dữ liệu có độ đa dạng cao.
- RGB(71.46%), Grayscale (70.90%) và LAB (70.23%) có đường cong gần như trùng nhau, cho thấy khả năng giữ lại thông tin ở các trục chính của PCA ở 3 không gian này cao tương đương nhau.
- Độ lệch chuẩn (std) của các không gian màu dao động từ 0.02 đến 0.03, cho thấy kết quả phân loại khá ổn định qua các lần folds


---
## (c) Chuẩn Hóa

### Lý thuyết

| Phương pháp | Công thức | Phạm vi |
|-------------|-----------|----------|
| **Min-Max [0,1]** | $x' = \dfrac{x - x_{\min}}{x_{\max} - x_{\min}}$ | $[0, 1]$ |
| **Min-Max [−1,1]** | $x' = 2\cdot\dfrac{x - x_{\min}}{x_{\max} - x_{\min}} - 1$ | $[-1, 1]$ |
| **Z-score (global)** | $x' = \dfrac{x - \mu}{\sigma}$ | $\mathbb{R}$ |
| **Z-score (per-channel)** | $x'_c = \dfrac{x_c - \mu_c}{\sigma_c}$ | $\mathbb{R}$ |

**Kiểm định Kolmogorov-Smirnov (KS test)**:
$$D = \sup_x |F_1(x) - F_2(x)|$$
$p < 0.05$ → hai phân phối khác nhau có ý nghĩa thống kê.

In [ ]:
import gc

gc.collect()

In [ ]:
def norm_minmax_01(X):
    mn = X.min(axis=1, keepdims=True)
    mx = X.max(axis=1, keepdims=True)
    return (X - mn) / (mx - mn + 1e-8)


def norm_minmax_11(X):
    return norm_minmax_01(X) * 2 - 1


def norm_zscore_global(X):
    return (X - X.mean()) / (X.std() + 1e-8)


def norm_zscore_perchannel(X, n_ch=3):
    """X shape (n, H*W*C) → per-channel z-score."""
    n = X.shape[0]
    imgs = X.reshape(n, n_ch, -1)
    mu = imgs.mean(axis=(0, 2), keepdims=True)
    std = imgs.std(axis=(0, 2), keepdims=True) + 1e-8
    return ((imgs - mu) / std).reshape(n, -1)

In [ ]:
X_raw_sample = images[SAMPLED_INDICES].reshape(N_SAMPLE, -1).astype(np.float64)

In [ ]:
X_raw_flat = X_raw_sample.ravel()

In [ ]:
normalization_configs = [
    ("Raw [0,1]", lambda x: x),
    ("MinMax [0,1]", norm_minmax_01),
    ("MinMax [-1,1]", norm_minmax_11),
    ("Z-score (global)", norm_zscore_global),
    ("Z-score (per-ch)", norm_zscore_perchannel),
]

In [ ]:
norm_results = {}
print(
    f'{"Method":22s} | {"Mean":>8s} | {"Std":>8s} | {"KS Stat":>9s} | {"LR Acc":>8s} | {"k-NN":>8s}'
)
print("-" * 82)
for name, func in normalization_configs:
    X_n = func(X_raw_sample).astype(np.float32)

    res = run_ablation(
        X_n,
        SAMPLED_LABELS,
        label=f"Norm {name}",
        n_pca=N_PCA,
        cv=CV,
        k_knn=K_KNN,
        C_lr=C_LOGREG,
        verbose=False,
    )
    register(res)

    ks_stat, pval = ks_2samp(X_raw_flat, X_n.ravel())
    norm_results[name] = {**res, "ks_stat": ks_stat, "pval": pval}

    print(
        f" {name:21s} | {X_n.mean():8.3f} | {X_n.std():8.3f} | "
        f'{ks_stat:9.5f} | {res["lr_acc"]:8.4f} | {res["knn_acc"]:8.4f}'
    )

    del X_n

In [ ]:
del X_raw_sample
del norm_results
gc.collect()

### Phân tích kết quả

* Raw [0,1]: Dữ liệu gốc có Mean ($\approx 113.6$) và Std ($\approx 70.6$) rất lớn. Điều này cho thấy các giá trị pixel trải dài trên dải rộng, dễ gây ra gradient explosion nếu dùng các mạng neuron hoặc làm chậm quá trình hội tụ của Logistic Regression.
* MinMax: Cả hai phương pháp ([0,1] và [-1,1]) đều đưa dữ liệu về đúng khoảng kỳ vọng. MinMax [-1,1] có Mean gần 0 hơn (-0.104), giúp dữ liệu cân bằng hơn quanh trục gốc.
* Z-score: Cả Global và Per-channel đều đưa Mean về xấp xỉ 0 và Std về đúng 1.000. Đây là trạng thái lý tưởng cho các thuật toán tối ưu hóa dựa trên gradient và PCA.

* Chỉ số KS Stat của tất cả các phương pháp đều rất cao (từ 0.974 đến 0.978)  cho thấy sau khi chuẩn hóa, phân phối của dữ liệu đã thay đổi cực kỳ đáng kể so với dữ liệu gốc. Điều này là hợp lý vì chúng ta đã thay đổi hoàn toàn thang đo (scale) của pixel.Vì KS Stat gần bằng 1, p-value chắc chắn sẽ rất nhỏ ($p < 0.001$), bác bỏ giả thuyết cho rằng hai phân phối là giống nhau. Điều này minh chứng rằng bước tiền xử lý đã có tác động thực sự lên dữ liệu.

---
## (d) Tăng Cường Dữ Liệu (Data Augmentation)

### Lý thuyết

Data augmentation mở rộng tập huấn luyện bằng cách tạo ra các biến thể của ảnh gốc mà **nhãn không thay đổi**. Mục tiêu: giảm overfitting, tăng tính bất biến của mô hình.

Pipeline gồm 6 phép biến đổi:
1. **Lật ngang** (Horizontal Flip) — $p=0.5$
2. **Lật dọc** (Vertical Flip) — $p=0.3$
3. **Xoay** (Rotation ±30°) — $p=0.5$
4. **Cắt ngẫu nhiên** (Random Crop 56×56 → resize 64×64) — $p=0.5$
5. **Nhiễu Gaussian** — $p=0.5$
6. **Điều chỉnh độ sáng/tương phản** — $p=0.5$

**t-SNE** chiếu dữ liệu nhiều chiều xuống 2D để trực quan hóa cấu trúc cụm, giữ nguyên khoảng cách cục bộ.

In [ ]:
import albumentations as A

In [ ]:
aug_pipeline = A.Compose(
    [
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.3),
        A.Rotate(limit=30, p=0.5),
        A.GaussNoise(std_range=(0.2, 1.0), p=0.5),
        A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
    ]
)


def augment_one(img_u8):
    return aug_pipeline(image=img_u8)["image"]


def augment_batch(imgs_f32):
    """augmented float32 [0,1]."""
    imgs_u8 = (imgs_f32 * 255).astype(np.uint8)
    out = np.stack([augment_one(img) for img in imgs_u8])
    return out.astype(np.float32) / 255.0

In [ ]:
images_aug = augment_batch(images[SAMPLED_INDICES])

In [ ]:
X_raw = images[SAMPLED_INDICES].reshape(N_SAMPLE, -1).astype(np.float64)

In [ ]:
X_aug = images_aug.reshape(N_SAMPLE, -1).astype(np.float64)

In [ ]:
print("\nAblation - Original:")
res_raw = run_ablation(X_raw, SAMPLED_LABELS, label="Original)")
register(res_raw)

In [ ]:
print("\nAblation - Augmented:")
res_aug = run_ablation(X_aug, SAMPLED_LABELS, label="Augmented")
register(res_aug)

In [ ]:
# t-SNE visualization (subsample 500)
TSNE_N = 500
tsne_idx = np.random.choice(len(SAMPLED_INDICES), TSNE_N, replace=False)
y_tsne = SAMPLED_LABELS[tsne_idx]


def pca_tsne(X_flat, n_pca=50, perplexity=30):
    pca = PCA(n_components=min(n_pca, X_flat.shape[1]), random_state=SEED)
    X_r = pca.fit_transform(X_flat)
    tsne = TSNE(n_components=2, perplexity=perplexity, random_state=SEED, max_iter=1000)
    return tsne.fit_transform(X_r)


print("Running t-SNE on original subset (n=500)...")
emb_orig = pca_tsne(X_raw[tsne_idx])
print("Running t-SNE on augmented subset (n=500)...")
emb_aug = pca_tsne(X_aug[tsne_idx])
print("Done.")

In [ ]:
CMAP = plt.colormaps.get_cmap("tab10")

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(
    "(d) t-SNE: Original vs Augmented Feature Distribution",
    fontsize=13,
    fontweight="bold",
)

for ax, emb, title in zip(
    axes, [emb_orig, emb_aug], ["Original", "After Augmentation"]
):
    for c in range(NUM_CLASSES):
        mask = y_tsne == c
        ax.scatter(
            emb[mask, 0],
            emb[mask, 1],
            c=[CMAP(c)],
            label=class_names[c],
            s=20,
            alpha=0.75,
            linewidths=0,
        )
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("t-SNE 1")
    ax.set_ylabel("t-SNE 2")
    ax.legend(markerscale=2, fontsize=8)
    ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.savefig("d_augmentation_tsne.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
del X_raw
del X_aug
del images_aug
gc.collect()

### Phân tích kết quả
Pipeline tăng cường dữ liệu bao gồm 5 phép biến đổi phi tuyến tính và hình học: Lật ngang/dọc (Horizontal/Vertical Flip), Xoay (Rotate), Thêm nhiễu Gaussian (GaussNoise), và Điều chỉnh độ sáng/tương phản (RandomBrightnessContrast).

**Phân tích tác động qua trực quan hóa t-SNE:**
- Tập gốc (Original): Các cụm dữ liệu có xu hướng tập trung hơn, một số lớp như forest (cam) và mountain (đỏ) tập trung thành các cụm tách biệt rõ rệt


**Kết quả ablation**: accuracy trên tập augmented thấp hơn. Accuracy của LogReg giảm từ 45.6% xuống 31.1%. Accuracy của k-NN giảm từ 41.3% xuống 30.1%. Dữ liệu sau tăng cường có độ phức tạp cao hơn đáng kể (thể hiện qua sự tăng độ lệch chuẩn và giảm accuracy baseline).

---
## (e) [Nâng cao] Phân Tích PCA Trên Không Gian Đặc Trưng Ảnh

### Lý thuyết

**PCA (Principal Component Analysis)** tìm các hướng có phương sai lớn nhất:
$$\mathbf{z} = \mathbf{W}^T (\mathbf{x} - \boldsymbol{\mu})$$

Trong đó $\mathbf{W}$ là ma trận các eigenvector của $\mathbf{\Sigma}$.

**Scree plot** vẽ phương sai giải thích theo số thành phần, xác định 'elbow'.

**Silhouette score**: $s(i) = \dfrac{b(i) - a(i)}{\max(a(i),\, b(i))} \in [-1, 1]$ — đo mức độ tách biệt các cụm.

In [ ]:
X_raw = images[SAMPLED_INDICES].reshape(N_SAMPLE, -1).astype(np.float64)
X_norm = norm_zscore_perchannel(X_raw)

In [ ]:
print("Fitting PCA")
pca_full = PCA(random_state=SEED)
pca_full.fit(X_norm)
cum_var = np.cumsum(pca_full.explained_variance_ratio_) * 100

thresholds = {}
for t in [90, 95, 99]:
    k = int(np.argmax(cum_var >= t)) + 1
    thresholds[t] = k
    print(f"  {t}% variance → {k} components")

print("\nLR ablation at variance thresholds:")
for t, k in thresholds.items():
    run_ablation(X_norm, SAMPLED_LABELS, n_pca=k, label=f"PCA-{k} ({t}% var)")

In [ ]:
pca2 = PCA(n_components=2, random_state=SEED)
X_2d = pca2.fit_transform(X_norm)
pca3 = PCA(n_components=3, random_state=SEED)
X_3d = pca3.fit_transform(X_norm)

tsne_pca_idx = np.random.choice(len(X_norm), 500, replace=False)
pca50 = PCA(n_components=50, random_state=SEED)
X_50 = pca50.fit_transform(X_norm[tsne_pca_idx])
print("Running t-SNE 2D (n=500)...")
X_tsne_pca = TSNE(
    n_components=2, perplexity=40, random_state=SEED, max_iter=1000
).fit_transform(X_50)
y_pca_sub = SAMPLED_LABELS[tsne_pca_idx]

In [ ]:
fig = plt.figure(figsize=(22, 12))
fig.suptitle(
    "(e) PCA Analysis - Scree Plot, 2D/3D Projections & t-SNE",
    fontsize=14,
    fontweight="bold",
)
gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.3)

# Scree plot
ax0 = fig.add_subplot(gs[0, :2])
ax0.plot(
    range(1, min(301, len(cum_var) + 1)),
    cum_var[:300],
    linewidth=1.5,
    color="steelblue",
)
colors_t = {90: "red", 95: "orange", 99: "green"}
for t, k in thresholds.items():
    ax0.axhline(t, color=colors_t[t], linestyle="--", alpha=0.7, label=f"{t}% @ k={k}")
    ax0.axvline(k, color=colors_t[t], linestyle=":", alpha=0.5)
ax0.set(
    title="Scree Plot - Cumulative Explained Variance",
    xlabel="# Components",
    ylabel="Cumulative Variance (%)",
)
ax0.set_xlim(0, 300)
ax0.set_ylim(0, 101)
ax0.legend(fontsize=9)
ax0.grid(True, alpha=0.3)

# Individual variance top-50
ax1 = fig.add_subplot(gs[0, 2])
ax1.bar(
    range(1, 51),
    pca_full.explained_variance_ratio_[:50] * 100,
    color="steelblue",
    alpha=0.75,
)
ax1.set(
    title="Individual Variance per Component (Top-50)",
    xlabel="Component",
    ylabel="Variance (%)",
)
ax1.grid(True, alpha=0.3)

# 2D PCA
ax2 = fig.add_subplot(gs[1, 0])
for c in range(NUM_CLASSES):
    # FIX: Use SAMPLED_LABELS instead of global labels
    m = SAMPLED_LABELS == c
    ax2.scatter(
        X_2d[m, 0],
        X_2d[m, 1],
        c=[CMAP(c)],
        label=class_names[c],
        s=8,
        alpha=0.5,
        linewidths=0,
    )
ev2d = pca2.explained_variance_ratio_.sum() * 100
ax2.set(title=f"PCA 2D  (EV={ev2d:.1f}%)", xlabel="PC1", ylabel="PC2")
ax2.legend(fontsize=7, markerscale=2)
ax2.grid(True, alpha=0.2)

# 3D PCA
ax3 = fig.add_subplot(gs[1, 1], projection="3d")
for c in range(NUM_CLASSES):
    m = SAMPLED_LABELS == c
    ax3.scatter(
        X_3d[m, 0],
        X_3d[m, 1],
        X_3d[m, 2],
        c=[CMAP(c)],
        label=class_names[c],
        s=6,
        alpha=0.4,
        linewidths=0,
    )
ax3.set(title="PCA 3D", xlabel="PC1", ylabel="PC2", zlabel="PC3")
ax3.legend(fontsize=6, markerscale=2)

# t-SNE
ax4 = fig.add_subplot(gs[1, 2])
for c in range(NUM_CLASSES):
    m = y_pca_sub == c
    ax4.scatter(
        X_tsne_pca[m, 0],
        X_tsne_pca[m, 1],
        c=[CMAP(c)],
        label=class_names[c],
        s=12,
        alpha=0.7,
        linewidths=0,
    )
ax4.set(title="t-SNE 2D (PCA-50 init, n=500)", xlabel="t-SNE 1", ylabel="t-SNE 2")
ax4.legend(fontsize=7, markerscale=2)
ax4.grid(True, alpha=0.2)

plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

sil_pca2 = silhouette_score(X_2d, SAMPLED_LABELS, sample_size=1000, random_state=SEED)
sil_tsne = silhouette_score(X_tsne_pca, y_pca_sub, sample_size=500, random_state=SEED)

print(f"Silhouette score - PCA-2D  (n=1200) : {sil_pca2:.4f}")
print(f"Silhouette score - t-SNE   (n=500)  : {sil_tsne:.4f}")
for t, k in thresholds.items():
    print(f"  {t}% variance -> {k} components")

In [ ]:
del X_raw
del X_norm
gc.collect()

### Phân tích kết quả

- Scree plot cho thấy phần lớn phương sai được giải thích bởi một số nhỏ thành phần đầu.
- PCA (2D/3D): Các điểm dữ liệu tạo thành một cụm lớn trung tâm overlap với nhau nhiều. PC1 và PC2 chỉ giải thích được 35.7% phương sai, không đủ để phân tách các lớp một cách rõ rệt. Tuy nhiên, có thể thấy lớp forest (màu cam) và glacier (màu xanh lá) có xu hướng lệch về các phía khác nhau của trục tọa độ.

- t-SNE (2D): Biểu đồ t-SNE cho thấy các cụm dữ liệu nhỏ nhưng các lớp vẫn đan xen vào nhau. Điều này cho thấy mối quan hệ giữa các pixel chưa đủ mạnh để phân cụm tốt.

- Silhouette score trên t-SNE > PCA-2D xác nhận các lớp tách biệt tốt hơn trong không gian t-SNE.

- Tại ngưỡng 90% (PCA-348): k-NN hoạt động tốt hơn LogReg (36.5% so với 32.3%). Ở mức nén này, dữ liệu còn ít nhiễu hơn, giúp khoảng cách hình học giữa các điểm lân cận có ý nghĩa hơn.

- Tại ngưỡng 99% (PCA-815): LogReg lại vượt lên dẫn đầu (37.5% so với 31.1%). Khi thêm nhiều thành phần PCA, LogReg dùng các chi tiết nhỏ để tìm ra siêu phẳng phân cách, trong khi k-NN bị ảnh hưởng bởi số lượng chiều dữ liệu lớn (curse of dimensionality), làm khoảng cách giữa các điểm trở nên kém phân biệt.
- Từ kết quả trên, ta thấy kết quả LogReg tăng dần theo số lượng thành phần PCA cho thấy mô hình này cần nhiều chi tiết hơn để tối ưu hóa hàm mất mát, trong khi k-NN đạt đỉnh sớm và suy giảm khi không gian quá rộng.

---
## (f) [Nâng cao] Phát Hiện Cạnh và Phân Tích Đặc Trưng Cục Bộ

### Lý thuyết

**Sobel operator** tính đạo hàm bậc nhất:
$$G_x = \begin{bmatrix}-1&0&1\\-2&0&2\\-1&0&1\end{bmatrix} * I, \quad G_y = G_x^T, \quad |G| = \sqrt{G_x^2 + G_y^2}$$

**Prewitt operator**: tương tự Sobel nhưng trọng số đồng đều hơn.

**Canny**: làm mịn Gaussian → Sobel → non-maximum suppression → hysteresis thresholding. Tham số $\sigma$ kiểm soát mức độ làm mịn.

**Edge Density**: $\text{ED} = \dfrac{\#\text{edge pixels}}{H \times W}$

**ANOVA một chiều** kiểm định $H_0$: edge density trung bình bằng nhau giữa các lớp. $p < 0.05$ → cạnh phân biệt được các lớp.

In [ ]:
def to_gray_batch(imgs):
    """(N,H,W,3) float32 -> (N,H,W) float64 grayscale."""
    return np.array([rgb2gray(img) for img in imgs])


imgs_gray = to_gray_batch(images[SAMPLED_INDICES])

EDGE_CFGS = {
    "Sobel (magnitude)": lambda g: np.hypot(sobel_h(g), sobel_v(g)),
    "Sobel (skimage)": lambda g: sobel(g),
    "Prewitt (magnitude)": lambda g: np.hypot(prewitt_h(g), prewitt_v(g)),
    "Prewitt (skimage)": lambda g: prewitt(g),
    "Canny (sigma=1.0)": lambda g: canny(g, sigma=1.0).astype(float),
    "Canny (sigma=2.0)": lambda g: canny(g, sigma=2.0).astype(float),
}

THRESHOLD = 0.05  # binarisation threshold cho Sobel/Prewitt

print("Computing edge density per image per method...")
edge_rows = []
for cfg_name, fn in EDGE_CFGS.items():
    for i, img_g in enumerate(imgs_gray):
        edge_map = fn(img_g)
        density = (
            edge_map.mean() if "Canny" in cfg_name else (edge_map > THRESHOLD).mean()
        )
        edge_rows.append(
            {
                "method": cfg_name,
                "label": labels[i],
                "class": class_names[labels[i]],
                "density": density,
            }
        )

edge_df = pd.DataFrame(edge_rows)
print(f"Done. edge_df shape: {edge_df.shape}")
print(edge_df.groupby(["method", "class"])["density"].mean().unstack().round(4))

In [ ]:
print(f'\n{"Method":25s} | {"F-stat":>10s} | {"p-value":>10s} | Significant?')
print("-" * 65)

anova_results = {}
for cfg_name in EDGE_CFGS:
    groups = [
        edge_df[(edge_df["method"] == cfg_name) & (edge_df["label"] == c)][
            "density"
        ].values
        for c in range(NUM_CLASSES)
    ]
    F, p = f_oneway(*groups)
    anova_results[cfg_name] = {"F": F, "p": p}
    sig = (
        "*** (p<0.001)"
        if p < 0.001
        else ("** (p<0.01)" if p < 0.01 else ("* (p<0.05)" if p < 0.05 else "ns"))
    )
    print(f"  {cfg_name:23s} | {F:10.2f} | {p:10.2e} | {sig}")

f_stats = {k: round(v["F"], 4) for k, v in anova_results.items()}
p_vals = {k: v["p"] for k, v in anova_results.items()}
print("\nF-statistics:", f_stats)
print("p-values    :", p_vals)

In [ ]:
fig, axes = plt.subplots(3, 2, figsize=(16, 16))
fig.suptitle(
    "(f) Edge Density per Class - Boxplots & ANOVA", fontsize=14, fontweight="bold"
)

for i, cfg_name in enumerate(EDGE_CFGS):
    ax = axes[i // 2][i % 2]
    pivot = [
        edge_df[(edge_df["method"] == cfg_name) & (edge_df["label"] == c)][
            "density"
        ].values
        for c in range(NUM_CLASSES)
    ]
    bp = ax.boxplot(pivot, patch_artist=True)
    for patch, c in zip(bp["boxes"], range(NUM_CLASSES)):
        patch.set_facecolor(CMAP(c))
        patch.set_alpha(0.75)
    ax.set_xticklabels(class_names, rotation=25, fontsize=8)
    F, p = anova_results[cfg_name]["F"], anova_results[cfg_name]["p"]
    sig_txt = (
        "***" if p < 0.001 else ("**" if p < 0.01 else ("*" if p < 0.05 else "ns"))
    )
    ax.set_title(f"{cfg_name}\nANOVA F={F:.1f}, p={p:.2e} {sig_txt}", fontsize=9)
    ax.set_ylabel("Edge Density")
    ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.savefig("f_edge_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
sample_gray = imgs_gray[0]
sample_rgb = images[SAMPLED_INDICES][0]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
fig.suptitle(
    "(f) Edge Detection - Visual Comparison (Sample Image)",
    fontsize=13,
    fontweight="bold",
)

sample_rgb_disp = (
    sample_rgb.astype(float) / 255.0 if sample_rgb.max() > 1.0 else sample_rgb
)

axes[0][0].imshow(np.clip(sample_rgb_disp, 0, 1))
axes[0][0].set_title("Original (RGB)")
axes[0][0].axis("off")
axes[1][0].imshow(sample_gray, cmap="gray")
axes[1][0].set_title("Grayscale")
axes[1][0].axis("off")

for j, (cfg_name, fn) in enumerate(EDGE_CFGS.items()):
    r, c_idx = divmod(j, 3)
    ax = axes[r][c_idx + 1]
    ax.imshow(fn(sample_gray), cmap="hot")
    ax.set_title(cfg_name, fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.savefig("f_edge_visual.png", dpi=150, bbox_inches="tight")
plt.show()

### Phân tích kết quả

- **ANOVA** với F lớn và p < 0.001 (***) trên tất cả 6 bộ detector xác nhận **edge density phân biệt được các lớp một cách có ý nghĩa thống kê**.
- Các lớp *buildings* và *street* có edge density cao nhất (nhiều cạnh thẳng, góc cạnh nhân tạo).
- Các lớp *forest* và *mountain* có edge density thấp hơn (cạnh hữu cơ, ít sắc nét).
- **Canny (σ=1.0)** phát hiện nhiều cạnh mịn nhất; **Canny (σ=2.0)** lọc bỏ nhiễu tốt hơn nhưng mất cạnh nhỏ.
- → Thông tin cạnh **có giá trị phân biệt lớp** và có thể dùng làm đặc trưng bổ sung.

- Sau khi trích xuất mật độ cạnh (Edge Density) bằng 3 thuật toán Sobel, Prewitt và Canny, mật độ cao nhất ở Sobel (0.55) và thấp nhất ở Canny (0.08 - 0.18).
- Các lớp đối tượng (tòa nhà, núi, rừng, biển...) có cấu trúc phức tạp tương đương nhau, dẫn đến mật độ cạnh phân phối rất đồng đều và không có sự khác biệt rõ rệt giữa các nhóm.
- Kết quả kiểm định ANOVA một chiều cho thấy toàn bộ các phương pháp đều có p-value > 0.05 cho thấy mật độ cạnh không phải là yếu tố phân ly giữa các lớp.
- Mật độ cạnh cục bộ (Local Edge Density) là một đặc trưng yếu cho bài toán này; thay vì chỉ dùng mật độ, cần các đặc trưng cao cấp hơn (như hướng cạnh hoặc hình dạng) để phân biệt đối tượng.
- Việc thay đổi $\sigma$ trong Canny (từ 1.0 lên 2.0) giúp lọc bớt nhiễu chi tiết nhưng vẫn không làm tăng khả năng tách biệt lớp, củng cố kết luận về tính tương đồng cấu trúc của tập dữ liệu.

---
## Tổng Kết Ablation Study

Bảng dưới đây tóm tắt kết quả của tất cả các cấu hình tiền xử lý đã thực nghiệm.

In [ ]:
summary_df = ablation_summary_table()
print(summary_df.to_string(index=False))
summary_df

In [ ]:
fig, ax = plt.subplots(figsize=(18, 5))
configs = summary_df["Config"].tolist()
lr_vals = [float(v) for v in summary_df["LogReg acc"].tolist()]
knn_vals = [float(v) for v in summary_df["k-NN acc"].tolist()]
x = np.arange(len(configs))
w = 0.35

ax.bar(
    x - w / 2,
    [v * 100 for v in lr_vals],
    w,
    label="LogReg",
    alpha=0.85,
    color="steelblue",
)
ax.bar(
    x + w / 2, [v * 100 for v in knn_vals], w, label="k-NN", alpha=0.85, color="crimson"
)
ax.set_xticks(x)
ax.set_xticklabels(configs, rotation=40, ha="right", fontsize=8)
ax.set_ylabel("CV Accuracy (%)")
ax.set_title("Ablation Study - All Configurations")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.savefig("summary_ablation.png", dpi=150, bbox_inches="tight")
plt.show()

### Kết luận tổng thể
- Kích thước ảnh tối ưu 64x64 vì LogReg đạt độ chính xác cao nhất (46.4%); việc tăng lên 128x128 không mang lại lợi ích đáng kể nhưng lại làm tăng gánh nặng tính toán.

- Nên giữ không gian RGB hoặc LAB (~45.4%)

- Phương pháp Min-Max [0, 1] hoặc [-1, 1] được hiệu quả vì giúp k-NN đạt độ chính xác cao nhất (41.2%) và giảm độ lệch chuẩn (std) xuống mức thấp nhất (0.0129).